# Validation of scenario reproduction

The ATAG Waypoint reports are dense and information-heavy, meaning many assumptions are explicit and quantified in the report itself. However, for scenario reproduction purposes with AeroMAPS, we may either try to approximate results by tweaking parameters, or directly feed on report data.

Relative to the 2nd Edition of the ATAG Waypoint report, the 3rd edition has fewer available data in the main report (meaning many missing inputs were filled with information from the 2nd edition). The traffic growth assumptions remained similar to the 2nd edition, as their central scenario had close agreement to observed traffic evolution. The technology scenarios were revised, where a T0 was included (the baseline is now seen as the 2019-rozen fuel efficiency), and the T5 was removed (delays on hydrogen aircraft development). Operational improvements are similar to the 2nd edition. SAF deployment was revised to reflect production delays, BUT their scale-up is now more intense to compensate for delays. Carbon offsets (MBMs) were also modified between editions, where the 2nd ed. considered MBMs to cap emissions at 2019 level until 2035 then reduce linearly until 2050, while the 3rd ed. does not cap the 2019 until 2035, and the emissions path from 2035-2050 is not linear anymore.

The transition from ATAG’s Waypoint 2050 2nd to 3rd edition reflects a shift from a policy-imposed pathway (cap at 2019 levels via offsets, then linear decline) to a feasibility-driven trajectory based on real deployment constraints. The pre-2035 cap was dropped because the underlying assumptions proved unrealistic, while mitigation now depends mainly on slower, uncertain scaling of SAF and new technologies. In parallel, International Civil Aviation Organization’s LTAG (2022) formalized this shift by adopting a non-binding net-zero 2050 goal without intermediate caps or enforcement, meaning no binding decision replaced the earlier cap logic; accountability is therefore indirect, increasingly driven by regional policies (e.g. EU ETS) and financial pressures rather than global regulation.

In [ ]:
%matplotlib widget


# The scenario ships with the package; copy it somewhere writable before running,
# so this notebook's outputs and regenerated inputs land in ./workdir rather than
# in the installed AeroMAPS.
from aeromaps.utils.scenarios import prepare_scenario
import gemseo as gm
import numpy as np
import yaml

from aeromaps import create_process, assemble_processes
from aeromaps.utils.functions import custom_logger_config

SCENARIO = prepare_scenario("atag_3rd_edition_full")

custom_logger_config(gm.configure_logger())

# The report's own digitised curves. They used to sit here as literals; they now
# live in report_data/atag_3rd_edition_figures.yaml so that this notebook and the
# MyST document read one copy rather than maintaining two. Values unchanged.
with open("../report_data/atag_3rd_edition_figures.yaml") as f:
    _report = yaml.safe_load(f)

central_years = _report["central_traffic"]["years"]
central_rpk = _report["central_traffic"]["values"]
atag_scenario_emissions = {
    name: (curve["years"], curve["values"])
    for name, curve in _report["technology_scenarios"].items()
}

## Traffic forecasts

As Air Traffic forecast is said not to have changed from edition 2 to edition 3, lets verify that with the central scenario.

We use the Compound Annual Growth Rates (CAGR) relative to Revenue Passenger Kilometers (RPK) shown using data extraction from images in the 2nd edition report, and compare them with the RPK from images in the 3rd edition.

In [ ]:
s1 = create_process(configuration_file=str(SCENARIO / "config_files" / "config_s1.yaml"))
s1.compute()

plot_rpk = s1.plot("revenue_passenger_kilometer")
plot_rpk.ax.plot(
    central_years, np.array(central_rpk) * 1e12, ":", label="Central forecast - report"
)
plot_rpk.ax.legend()

Low and high traffic use the same central-year anchor and diverge afterwards (Phase 2's `alpha * central_growth(t)` construction), rather than a digitised report curve -- the third edition does not publish separate low/high traffic trajectories the way the second edition did.

In [ ]:
traffic_variants = assemble_processes(
    {
        "Low": create_process(
            configuration_file=str(SCENARIO / "config_files" / "config_s1-traffic-low.yaml")
        ),
        "Central": s1,
        "High": create_process(
            configuration_file=str(SCENARIO / "config_files" / "config_s1-traffic-high.yaml")
        ),
    }
)
traffic_variants.compute_all()
for process in traffic_variants:
    process.write_json()

traffic_variants.plot("rpk_comparison")

## Technology (full lifecycle)

Aircraft technology has significant improved since the commercialization of air travel, and efficiency gains have had a significant impact in reducing associated emissions.

Looking ahead, aircraft technology may continue with ever-modest efficiency gains, or adhere to disruptive technologies. We extract the CO2 emissions trajectory of 5 technological scenarios and fit annual efficiency gains based on these curbs.

In [ ]:
tech_variants = assemble_processes(
    {
        f"T{i}": create_process(
            configuration_file=str(SCENARIO / "config_files" / f"config_t{i}.yaml")
        )
        for i in range(5)
    }
)
tech_variants.compute_all()

# Persist these runs: their data_outputs/*.json are committed and read by the
# MyST document, but nothing else regenerates them, so they silently go stale
# whenever an input changes unless they are written here.
for process in tech_variants:
    process.write_json()

plot_co2 = tech_variants.plot("co2_emissions_comparison")
for i, process in enumerate(tech_variants):
    scenario_name = tech_variants.get_scenario_names()[i]
    plot_co2.ax.plot(
        atag_scenario_emissions[scenario_name][0],
        atag_scenario_emissions[scenario_name][1],
        ":",
        label=f"{scenario_name} - report",
    )
plot_co2.ax.legend()

In [ ]:
years_interp = np.arange(2024, 2051)

plot_co2_per_rpk = tech_variants.plot("co2_per_rpk_comparison")
for i, process in enumerate(tech_variants):
    scenario_name = tech_variants.get_scenario_names()[i]
    data_rpk = np.interp(years_interp, central_years, central_rpk)
    data_co2 = np.interp(
        years_interp,
        atag_scenario_emissions[scenario_name][0],
        atag_scenario_emissions[scenario_name][1],
    )
    plot_co2_per_rpk.ax.plot(
        years_interp, data_co2 / data_rpk, ":", label=f"{scenario_name} - report"
    )
plot_co2.ax.legend()


Compared to the 2nd edition, in the 3rd ed. all scenarios were revised with an additional 0.2% annual reduction in fuel burn in the 2019-2035 period, and in the T3 scenario (new configurations) after 2045 fuel burn annual reduction is lower (meaning aircraft will be less efficient) by 0.4%.

## Operations

The operations lever is a cumulative efficiency gain applied uniformly across the fleet, reaching 0.00 / 0.10 / 0.20 %/yr by 2050 for O1/O2/O3 -- values read directly from the report text (the highest-confidence provenance tier). S1 already runs at O3; O1 and O2 are computed here for comparison.

In [ ]:
ops_variants = assemble_processes(
    {
        "O1 (0.00 %/yr)": create_process(
            configuration_file=str(SCENARIO / "config_files" / "config_s1.yaml")
        ),
        "O2 (0.10 %/yr)": create_process(
            configuration_file=str(SCENARIO / "config_files" / "config_s1.yaml")
        ),
        "O3 (0.20 %/yr, as published)": s1,
    }
)
ops_variants["O1 (0.00 %/yr)"].parameters.operations_gain_reference_years = [2020, 2050]
ops_variants["O1 (0.00 %/yr)"].parameters.operations_gain_reference_years_values = [0, 0]
ops_variants["O2 (0.10 %/yr)"].parameters.operations_gain_reference_years = [2020, 2050]
ops_variants["O2 (0.10 %/yr)"].parameters.operations_gain_reference_years_values = [0, 3]

ops_variants.compute_all()
ops_variants["O1 (0.00 %/yr)"].write_json(file_name="data_outputs/s1-ops-o1.json")
ops_variants["O2 (0.10 %/yr)"].write_json(file_name="data_outputs/s1-ops-o2.json")

ops_variants.plot("co2_emissions_comparison")

## Technology (CORSIA-like)

In [ ]:
tech_variants_corsia = assemble_processes(
    {
        f"T{i}": create_process(
            configuration_file=str(SCENARIO / "config_files" / f"config_t{i}-TTW.yaml")
        )
        for i in range(5)
    }
)
tech_variants_corsia.compute_all()

# Persist these runs: their data_outputs/*.json are committed and read by the
# MyST document, but nothing else regenerates them, so they silently go stale
# whenever an input changes unless they are written here.
for process in tech_variants_corsia:
    process.write_json()

plot_co2_corsia = tech_variants_corsia.plot("co2_emissions_comparison")
for i, process in enumerate(tech_variants):
    scenario_name = tech_variants.get_scenario_names()[i]
    plot_co2_corsia.ax.plot(
        atag_scenario_emissions[scenario_name][0],
        atag_scenario_emissions[scenario_name][1],
        ":",
        label=f"{scenario_name} - report",
    )
plot_co2_corsia.ax.legend()

In [ ]:
plot_co2_per_rpk_corsia = tech_variants.plot("co2_per_rpk_comparison")
for i, process in enumerate(tech_variants):
    scenario_name = tech_variants.get_scenario_names()[i]
    data_rpk = np.interp(years_interp, central_years, central_rpk)
    data_co2 = np.interp(
        years_interp,
        atag_scenario_emissions[scenario_name][0],
        atag_scenario_emissions[scenario_name][1],
    )
    plot_co2_per_rpk_corsia.ax.plot(
        years_interp, data_co2 / data_rpk, ":", label=f"{scenario_name} - report"
    )
plot_co2_corsia.ax.legend()

In [ ]:
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals(), force_cleanup=False)